1. Load dataset
2. eng->sentence-> embedding
3. build RNN
4. train
5. prediction


In [261]:
import pandas as pd

df = pd.read_csv('data/100_Unique_QA_Dataset.csv')

In [262]:
df.head()

,question,answer
0,What is the capital of France?,Paris
1,What is the capital of Germany?,Berlin
2,Who wrote 'To Kill a Mockingbird'?,Harper-Lee
3,What is the largest planet in our solar system?,Jupiter
4,What is the boiling point of water in Celsius?,100


In [263]:
#tokenize
def tokenize(text):
    text = text.lower()
    text = text.replace("?", '')
    text = text.replace("'", "")

    return text.split()

In [264]:
tokenize("what is my name?")

['what', 'is', 'my', 'name']

In [265]:
vocab = {'<UNK>' : 0}

In [266]:
def build_vocab(row):
    tokenized_question = tokenize(row['question'])
    tokenized_answer = tokenize(row['answer'])

    merged_tokens = tokenized_question + tokenized_answer 

    for token in merged_tokens:
        if token not in vocab:
            vocab[token] = len(vocab)

In [267]:
df.apply(build_vocab, axis=1)

0     None
1     None
2     None
3     None
4     None
      ... 
85    None
86    None
87    None
88    None
89    None
Length: 90, dtype: object

In [268]:
len(vocab)
vocab

{'<UNK>': 0,
 'what': 1,
 'is': 2,
 'the': 3,
 'capital': 4,
 'of': 5,
 'france': 6,
 'paris': 7,
 'germany': 8,
 'berlin': 9,
 'who': 10,
 'wrote': 11,
 'to': 12,
 'kill': 13,
 'a': 14,
 'mockingbird': 15,
 'harper-lee': 16,
 'largest': 17,
 'planet': 18,
 'in': 19,
 'our': 20,
 'solar': 21,
 'system': 22,
 'jupiter': 23,
 'boiling': 24,
 'point': 25,
 'water': 26,
 'celsius': 27,
 '100': 28,
 'painted': 29,
 'mona': 30,
 'lisa': 31,
 'leonardo-da-vinci': 32,
 'square': 33,
 'root': 34,
 '64': 35,
 '8': 36,
 'chemical': 37,
 'symbol': 38,
 'for': 39,
 'gold': 40,
 'au': 41,
 'which': 42,
 'year': 43,
 'did': 44,
 'world': 45,
 'war': 46,
 'ii': 47,
 'end': 48,
 '1945': 49,
 'longest': 50,
 'river': 51,
 'nile': 52,
 'japan': 53,
 'tokyo': 54,
 'developed': 55,
 'theory': 56,
 'relativity': 57,
 'albert-einstein': 58,
 'freezing': 59,
 'fahrenheit': 60,
 '32': 61,
 'known': 62,
 'as': 63,
 'red': 64,
 'mars': 65,
 'author': 66,
 '1984': 67,
 'george-orwell': 68,
 'currency': 69,
 'unit

In [269]:
def text_to_indices(text, vocab):
    indexed_text = []

    for token in tokenize(text):
        if token in vocab:
            indexed_text.append(vocab[token])
        else:
            indexed_text.append(vocab['<UNK>'])
    return indexed_text

In [270]:
text_to_indices("what is my name?", vocab)

[1, 2, 0, 0]

In [271]:
import torch
from torch.utils.data import Dataset, DataLoader

In [272]:
class QADataset(Dataset):
    def __init__(self, df, vocab):
        self.df = df
        self.vocab = vocab

    def __len__(self):
        return self.df.shape[0]
    
    def __getitem__(self, index):
        numberical_question = text_to_indices(self.df.iloc[index]['question'], self.vocab)
        numberical_answer = text_to_indices(self.df.iloc[index]['answer'], self.vocab)

        return torch.tensor(numberical_question), torch.tensor(numberical_answer)

In [273]:
dataset = QADataset(df, vocab)

In [274]:
dataloader = DataLoader(dataset, batch_size=1, shuffle=True)


In [275]:
for question, answer in dataloader:
    print(question, answer)

tensor([[10, 11, 12, 13, 14, 15]]) tensor([[16]])
tensor([[ 10,  75, 111]]) tensor([[112]])
tensor([[  1,   2,   3,   4,   5, 286]]) tensor([[287]])
tensor([[ 1,  2,  3,  4,  5, 53]]) tensor([[54]])
tensor([[10, 29,  3, 30, 31]]) tensor([[32]])
tensor([[ 78,  79, 261, 151,  14, 262, 153]]) tensor([[36]])
tensor([[  1,   2,   3,   4,   5, 135]]) tensor([[136]])
tensor([[ 1,  2,  3, 33, 34,  5, 35]]) tensor([[36]])
tensor([[ 10,  96,   3, 104, 239]]) tensor([[240]])
tensor([[  1,   2,   3,   4,   5, 109]]) tensor([[317]])
tensor([[  1,   2,   3,  33,  34,   5, 245]]) tensor([[246]])
tensor([[ 42, 299, 300, 118,  14, 301, 302, 158, 303, 304, 305, 306]]) tensor([[307]])
tensor([[ 10,   2,  62,  63,   3, 283,   5, 284]]) tensor([[285]])
tensor([[ 10,  11, 157, 158, 159]]) tensor([[160]])
tensor([[ 42, 255,   2, 256,  83, 257, 258]]) tensor([[259]])
tensor([[78, 79, 80, 81, 82, 83, 84]]) tensor([[85]])
tensor([[42, 43, 44, 45, 46, 47, 48]]) tensor([[49]])
tensor([[ 1,  2,  3, 17, 18, 19, 20,

In [276]:
import torch.nn as nn 

class SimpleRNN(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim=50)

        self.rnn = nn.RNN(50, 64, batch_first=True)
        self.fc = nn.Linear(64, vocab_size)

    def forward(self, question):
        embedded_question = self.embedding(question)
        hidden, final = self.rnn(embedded_question)
        output = self.fc(final.squeeze(0))

        return output



In [277]:
x = nn.Embedding(324, embedding_dim=50)
y = nn.RNN(50,64, batch_first=True)
z = nn.Linear(64,324)

a = dataset[0][0].reshape(1,6)
print("shape of a:", a.shape)

b = x(a)
print("Shape of b: ", b.shape)

c,d = y(b)
print("shape of c: ", c.shape)
print("shape of d: ", d.shape)

shape of a: torch.Size([1, 6])
Shape of b:  torch.Size([1, 6, 50])
shape of c:  torch.Size([1, 6, 64])
shape of d:  torch.Size([1, 1, 64])


In [278]:
learning_rate = 0.001
epoch = 20

In [279]:
model = SimpleRNN(len(vocab))

In [280]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [281]:
for epoch in range(epoch):
    total_loss = 0

    for question, answer in dataloader:
        optimizer.zero_grad()

        output = model(question)

        loss = criterion(output, answer[0])

        loss.backward()

        optimizer.step()

        total_loss = total_loss + loss.item()

    print(f"Epoch: {epoch+1}, Loss: {total_loss:4f}")

Epoch: 1, Loss: 522.947771
Epoch: 2, Loss: 459.096471
Epoch: 3, Loss: 383.926092
Epoch: 4, Loss: 316.314380
Epoch: 5, Loss: 263.018976
Epoch: 6, Loss: 214.784733
Epoch: 7, Loss: 170.752306
Epoch: 8, Loss: 133.820014
Epoch: 9, Loss: 102.146182
Epoch: 10, Loss: 78.133200
Epoch: 11, Loss: 59.674848
Epoch: 12, Loss: 47.077650
Epoch: 13, Loss: 37.599414
Epoch: 14, Loss: 30.218229
Epoch: 15, Loss: 25.051103
Epoch: 16, Loss: 20.981462
Epoch: 17, Loss: 17.521141
Epoch: 18, Loss: 15.078095
Epoch: 19, Loss: 12.900289
Epoch: 20, Loss: 11.187431


In [283]:
def predict(model, question, threshold=0.5):
    #convert question to numbers
    numberical_question = text_to_indices(question, vocab)

    # tensor 
    question_tensor = torch.tensor(numberical_question).unsqueeze(0)
    with torch.no_grad():

        #send to model 
        output = model(question_tensor)
        
        #convert logits to probs

        probs = torch.nn.functional.softmax(output, dim=1)

        # find index of max prob 

        value, index = torch.max(probs, dim=1)

        if value < threshold:
            print("I dont know")

        print(list(vocab.keys())[index])


In [284]:
predict(model, "What is the largest planet in our solar system?")

jupiter
